# **What new dataset shows**

1. If accuracy will drop on 70-80% it means that model overtrained on phishing dataset from hugging face
2. If accuracy will stay high, model is probably really solid

In [12]:
import pandas as pd
import numpy as np

import matplotlib.pyplot as plt
import seaborn as sns

from sklearn.model_selection import train_test_split

import sys
sys.path.append('..')
from utils import EmailDataset, evaluate_model

from transformers import AutoTokenizer, AutoModelForSequenceClassification, Trainer, TrainingArguments
import torch
from torch.utils.data import Dataset

from sklearn.metrics import accuracy_score, f1_score, precision_score, recall_score, roc_auc_score, classification_report


In [2]:
device = torch.device('cuda' if torch.cuda.is_available() else 'cpu')
print(device)

cuda


In [3]:
df = pd.read_csv('phishing_email.csv')

In [4]:
df.head()

,text_combined,label
0,hpl nom may 25 2001 see attached file hplno 52...,0
1,nom actual vols 24 th forwarded sabrae zajac h...,0
2,enron actuals march 30 april 1 201 estimated a...,0
3,hpl nom may 30 2001 see attached file hplno 53...,0
4,hpl nom june 1 2001 see attached file hplno 60...,0


In [5]:
df.shape

(80106, 2)

In [6]:
df.info()

<class 'pandas.core.frame.DataFrame'>
RangeIndex: 80106 entries, 0 to 80105
Data columns (total 2 columns):
 #   Column         Non-Null Count  Dtype 
---  ------         --------------  ----- 
 0   text_combined  80106 non-null  object
 1   label          80106 non-null  int64 
dtypes: int64(1), object(1)
memory usage: 1.2+ MB


In [7]:
X = df['text_combined']
y = df['label']

In [8]:
_ , X_test, _ , y_test = train_test_split(
    X,
    y,
    random_state=42,
    test_size=0.3,
    stratify=y
)

In [15]:
# transform datasets
bert_test_dataset_v2 = EmailDataset(X_test, y_test, bert_tokenizer)
cysec_test_dataset_v2 = EmailDataset(X_test, y_test, cysec_tokenizer)

In [9]:
bert_tokenizer  = AutoTokenizer.from_pretrained('../saved_models/bert')
cysec_tokenizer = AutoTokenizer.from_pretrained('../saved_models/cysec')

In [13]:
bert_model  = AutoModelForSequenceClassification.from_pretrained('../saved_models/bert').to(device)
cysec_model = AutoModelForSequenceClassification.from_pretrained('../saved_models/cysec').to(device)


Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

Loading weights:   0%|          | 0/201 [00:00<?, ?it/s]

In [16]:
args = TrainingArguments(output_dir='./results/datavest_v2', per_device_eval_batch_size=32)

bert_trainer  = Trainer(model=bert_model,  args=args)
cysec_trainer = Trainer(model=cysec_model, args=args)

In [18]:
bert_results  = evaluate_model(bert_trainer,  bert_test_dataset_v2,  'BERT on new  dataset')
cysec_results = evaluate_model(cysec_trainer, cysec_test_dataset_v2, 'CySecBERT on new dataset')


Model: BERT on new  dataset
Accuracy:  0.7830
F1-score:  0.8181
Precision: 0.7101
Recall:    0.9648
ROC-AUC:   0.9325

Detailed report:
              precision    recall  f1-score   support

        Safe       0.94      0.60      0.73     11879
    Phishing       0.71      0.96      0.82     12153

    accuracy                           0.78     24032
   macro avg       0.83      0.78      0.77     24032
weighted avg       0.83      0.78      0.78     24032




Model: CySecBERT on new dataset
Accuracy:  0.7766
F1-score:  0.8179
Precision: 0.6957
Recall:    0.9923
ROC-AUC:   0.9255

Detailed report:
              precision    recall  f1-score   support

        Safe       0.99      0.56      0.71     11879
    Phishing       0.70      0.99      0.82     12153

    accuracy                           0.78     24032
   macro avg       0.84      0.77      0.76     24032
weighted avg       0.84      0.78      0.77     24032

